# Uninsured rate - pull, clean, export

Indicator: percent uninsured, civilian noninstitutionalized population. Census ACS5 Subject Table
S2701, variable S2701_C05_001E. Geographies: Allegheny County, Pennsylvania, United States.

Same `CENSUS_API_KEY` as the SNAP pull - no new secret needed.

In [1]:
import json
import os
import time
from datetime import date
from pathlib import Path

import pandas as pd
import requests

API_KEY = os.environ.get("CENSUS_API_KEY")
if not API_KEY:
    raise RuntimeError("CENSUS_API_KEY not set. Set it as an environment variable, do not hardcode it here.")

RAW_DIR = Path("../raw")
DATA_DIR = Path("../data")
RAW_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

today = date.today().isoformat()
print("Pipeline run date:", today)

Pipeline run date: 2026-10-04


In [2]:
VARIABLE = "S2701_C05_001E"  # Percent uninsured, civilian noninstitutionalized population, total
DATASET = "acs/acs5/subject"
YEARS = [2014, 2016, 2018, 2020, 2022]  # matches the SNAP module's spacing

GEOGRAPHIES = {
    "Allegheny County": {"for": "county:003", "in": "state:42"},
    "Pennsylvania": {"for": "state:42"},
    "United States": {"for": "us:1"},
}

In [3]:
rows = []
raw_responses = {}

for year in YEARS:
    url = f"https://api.census.gov/data/{year}/{DATASET}"
    for geo_name, geo_params in GEOGRAPHIES.items():
        params = {"get": f"NAME,{VARIABLE}", "key": API_KEY, **geo_params}
        try:
            resp = requests.get(url, params=params, timeout=30)
            resp.raise_for_status()
            payload = resp.json()
        except Exception as exc:
            print(f"FAILED  year={year}  geo={geo_name}  error={exc}")
            continue

        raw_responses[f"{year}_{geo_name}"] = payload

        header, data_row = payload[0], payload[1]
        record = dict(zip(header, data_row))
        value_raw = record.get(VARIABLE)

        value = None
        if value_raw is not None:
            try:
                value_f = float(value_raw)
                value = None if value_f < 0 else value_f
            except ValueError:
                value = None

        rows.append({"year": year, "geography": geo_name, "value": value})
        time.sleep(0.2)

with open(RAW_DIR / f"uninsured_rate_raw_{today}.json", "w") as f:
    json.dump(raw_responses, f, indent=2)

df = pd.DataFrame(rows)
df

,year,geography,value
0,2014,Allegheny County,92.5
1,2014,Pennsylvania,90.5
2,2014,United States,85.8
3,2016,Allegheny County,6.0
4,2016,Pennsylvania,8.0
5,2016,United States,11.7
6,2018,Allegheny County,4.3
7,2018,Pennsylvania,6.2
8,2018,United States,9.4
9,2020,Allegheny County,3.8


In [4]:
expected = len(YEARS) * len(GEOGRAPHIES)
print(f"Expected {expected} rows, got {len(df)}")
print(df[df["value"].isna()])

Expected 15 rows, got 15
Empty DataFrame
Columns: [year, geography, value]
Index: []


In [5]:
wide = df.pivot(index="year", columns="geography", values="value").reset_index()
series = wide.to_dict(orient="records")

payload = {
    "indicator": "Uninsured rate (% of civilian noninstitutionalized population)",
    "geographies": list(GEOGRAPHIES.keys()),
    "unit": "percent",
    "snapshot_as_of": today,
    "status": "live",
    "source": f"U.S. Census Bureau, American Community Survey 5-Year Estimates, Subject Table S2701, variable {VARIABLE}",
    "series": series,
    "notes": "",
}

with open(DATA_DIR / "uninsured_rate.json", "w") as f:
    json.dump(payload, f, indent=2)

print("Wrote", DATA_DIR / "uninsured_rate.json")
payload

Wrote ../data/uninsured_rate.json


{'indicator': 'Uninsured rate (% of civilian noninstitutionalized population)',
 'geographies': ['Allegheny County', 'Pennsylvania', 'United States'],
 'unit': 'percent',
 'snapshot_as_of': '2026-10-04',
 'status': 'live',
 'source': 'U.S. Census Bureau, American Community Survey 5-Year Estimates, Subject Table S2701, variable S2701_C05_001E',
 'series': [{'year': 2014,
   'Allegheny County': 92.5,
   'Pennsylvania': 90.5,
   'United States': 85.8},
  {'year': 2016,
   'Allegheny County': 6.0,
   'Pennsylvania': 8.0,
   'United States': 11.7},
  {'year': 2018,
   'Allegheny County': 4.3,
   'Pennsylvania': 6.2,
   'United States': 9.4},
  {'year': 2020,
   'Allegheny County': 3.8,
   'Pennsylvania': 5.6,
   'United States': 8.7},
  {'year': 2022,
   'Allegheny County': 3.9,
   'Pennsylvania': 5.6,
   'United States': 8.7}],
 'notes': ''}